In [2]:
# Load the indian dataset and convert it to be of the same style as tab

import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'data/train-00000-of-00001-de25c1ae1db42f79.parquet', 'dev': 'data/dev-00000-of-00001-b148266485fd7aeb.parquet', 'test': 'data/test-00000-of-00001-814a730def5e8488.parquet'}
df = pd.read_parquet("hf://datasets/opennyaiorg/InLegalNER/" + splits["train"])

/home/amandus/miniconda3/envs/leakpro/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
len(df)

10995

In [32]:
print(df.iloc[0]['data'])



{'text': "\n\n(7) On specific query by the Bench about an entry of Rs. 1,31,37,500 on deposit side of Hongkong Bank account of which a photo copy is appearing at p. 40 of assessee's paper book, learned authorised representative submitted that it was related to loan from broker, Rahul & Co. on the basis of his submission a necessary mark is put by us on that photo copy."}


In [33]:
ind_to_tab = {
    "COURT": "ORG",
    "PETITIONER":  "PERSON",
    "RESPONDENT": "PERSON",
    "JUDGE": "PERSON",
    "LAWYER": "PERSON",
    "DATE": "DATETIME",
    "ORG":  "ORG",
    "GPE":  "LOC",
    "STATUTE": "MISC",
    "PROVISION":  "MISC",
    "PRECEDENT": "MISC",
    "CASE_NUMBER": "CODE",
    "WITNESS": "PERSON",
    "OTHER_PERSON":  "PERSON"
}

training_raw = []
for i in range(len(df)):
    dct = {}
    dct['split'] = 'train'
    dct['text'] = df.iloc[i]['data']['text']
    dct['doc_id'] = i


    annotations = df.iloc[i]["annotations"][0]['result']
    anno_list = []
    for j in range(len(annotations)):
        entity_type = annotations[j]['value']['labels'][0]
        span_text = annotations[j]['value']['text']
        label = 'MASK'
        identifier_type = "QUASI"
        entity_id = annotations[j]['id']
        start_offset = annotations[j]['value']['start']
        end_offset = annotations[j]['value']['end']
        anno_list.append(
            {"entity_type": ind_to_tab[entity_type], 
            "span_text": span_text,
            'start_offset': start_offset,
            'end_offset': end_offset,
            "identifier_type": identifier_type,
            "entity_id" : entity_id,
            "label": label,
            "id":i
            }
        )
    dct['annotations'] = anno_list
    training_raw.append(dct)


In [8]:
import json
with open('/home/amandus/msc-code/text-anonymization-benchmark/echr_train.json', "r", encoding="utf-8") as f1:

    train = json.load(f1)

    training_raw2 = []
    dev_raw = []
    test_raw = []
    
    for ann_data in train:
        dct = {}
        dct['split'] = ann_data['dataset_type']
        dct['text'] = ann_data['text']
        dct['doc_id'] = ann_data['doc_id']
        did = ann_data['doc_id']
        for annotator in ann_data['annotations']:
            annotations = []
            for annotation in ann_data['annotations'][annotator]['entity_mentions']:
                if annotation['identifier_type'] != 'NO_MASK':
                    annotation['label'] = 'MASK'
                    #annotation['entity_type'] = annotation['entity_type']
                else:
                    annotation['label'] = 'NO_MASK'
                annotation['id'] = did
                annotation['span_text'] = annotation['span_text']
                annotations.append(annotation)

            dct['annotations'] = annotations
            training_raw2.append(dct)
 

In [14]:
training_raw2[0].keys()

dict_keys(['split', 'text', 'doc_id', 'annotations'])

In [24]:
for key in training_raw[0].keys():
    print("new key-------")
    print("tab", training_raw2[0][key])
    print('-----')
    print("indian", training_raw[0][key])
    print()

new key-------
tab train
-----
indian train

new key-------
tab PROCEDURE

The case originated in an application (no. 36244/06) against the Kingdom of Denmark lodged with the Court under Article 34 of the Convention for the Protection of Human Rights and Fundamental Freedoms (“the Convention”) by a Danish national, Mr Henrik Hasslund (“the applicant”), on 31 August 2006.

The applicant was represented by Mr Tyge Trier, a lawyer practising in Copenhagen. The Danish Government (“the Government”) were represented by their Agent, Ms Nina Holst-Christensen of the Ministry of Justice.

On 5 September 2007 the Acting President of the Fifth Section decided to give notice of the application to the Government. It was also decided to rule on the admissibility and merits of the application at the same time (Article 29 § 3).

THE FACTS

THE CIRCUMSTANCES OF THE CASE

The applicant was born in 1973 and lives in Les Salles Sur Verdon, France.

At the beginning of the 1990s a new concept called “tax a

In [34]:
import pickle
with open("./indian_ner_small_raw.pkl", "wb") as handle:
    pickle.dump(training_raw[0:2000], handle)